# 05c · Modèle 3 : segmentation finale (expérience, valeur et grille d'actions)
**Projet : E-Commerce Intelligence Platform** · CRISP-DM, phase 5 (Evaluation), avant le déploiement

**Pourquoi ce notebook ?** Le rapport d'évaluation a montré que les trois segments obtenus (satisfaits 78 %, insatisfaits sans retard 16 %, insatisfaits par la livraison 6 %) se distinguent **uniquement par l'expérience** : le montant et la récence sont quasi identiques d'un segment à l'autre. Les segments ne permettent donc pas de cibler par valeur.

**Ce que fait ce notebook**
1. Teste **trois variantes** de segmentation : **V1** expérience complète (6 variables, celle du rapport), **V2** comportement seul (sans la note), **V3** valeur, récence et satisfaction (sans les variables de livraison).
2. Mesure ce que chaque variante **capture** : part de la variance de la récence, de la valeur, de la satisfaction et de la livraison expliquée par les segments (η²), en plus de la silhouette, de Davies-Bouldin et de la **stabilité**.
3. Retient la segmentation **d'expérience** (V1, stable et interprétable) et lui ajoute une **dimension de valeur** (terciles du montant dépensé) et de récence. On obtient une **grille d'actions** « expérience × valeur » qui répond au besoin de ciblage.
4. Écrit les fichiers pour le tableau de bord.

> Données : dataset Kaggle `e-commerce-dataset-model-3`. Durée indicative : 3 à 8 minutes. Fichiers écrits dans `/kaggle/working/deploy/`.

## 1. Imports et paramètres

In [ ]:
import os, glob, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

SEED = 42
rng = np.random.default_rng(SEED)
FAST = os.environ.get("FAST", "0") == "1"
OUT_DIR = os.environ.get("OUT_DIR", "/kaggle/working/deploy")
os.makedirs(OUT_DIR, exist_ok=True)

K_RANGE = range(2, 7) if FAST else range(2, 9)
CLIP_Q = 0.005
SIL_SAMPLE = 3000 if FAST else 15000
N_INIT = 3 if FAST else 10
N_BOOT = 2 if FAST else 5
K_OVERRIDE = {}              # par exemple {"V2": 4} pour forcer k d'une variante
FINAL_VARIANT = "V1"         # segmentation d'expérience retenue (modifiable)

# Chemins des jeux de données Kaggle (créés à partir de la sortie du notebook 03)
DATA_PATHS = {
    "model1_delivery_duration.csv": "/kaggle/input/datasets/oussamafriha/e-commerce-dataset-model-1/model1_delivery_duration.csv",
    "model2_late_risk.csv": "/kaggle/input/datasets/oussamafriha/e-commerce-dataset-model-2/model2_late_risk.csv",
    "model3_customer_segmentation.csv": "/kaggle/input/datasets/oussamafriha/e-commerce-dataset-model-3/model3_customer_segmentation.csv",
}

def find_file(name):
    """DATA_DIR (si défini), puis le chemin Kaggle ci-dessus, puis une recherche dans /kaggle/input."""
    cands = []
    if os.environ.get("DATA_DIR"):
        cands.append(os.path.join(os.environ["DATA_DIR"], name))
    cands.append(DATA_PATHS.get(name, ""))
    cands += glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    for c in cands:
        if c and os.path.exists(c):
            return c
    raise FileNotFoundError(f"{name} introuvable : ajoute le dataset correspondant en entrée du notebook (Add Input).")

## 2. Chargement

In [ ]:
m3 = pd.read_csv(find_file("model3_customer_segmentation.csv"), parse_dates=["first_purchase", "last_purchase"])
ALL = ["recency_days", "log_monetary", "avg_review_score", "avg_delivery_days", "log_max_days_late", "avg_gap_days"]
VARIANTS = {
    "V1": {"nom": "Expérience complète (6 variables)", "features": ALL},
    "V2": {"nom": "Comportement seul (sans la note)", "features": [c for c in ALL if c != "avg_review_score"]},
    "V3": {"nom": "Valeur, récence et satisfaction (sans livraison)", "features": ["recency_days", "log_monetary", "avg_review_score"]},
}
assert not [c + "_rs" for c in ALL if c + "_rs" not in m3.columns], "Colonnes mises à l'échelle absentes"
print(f"{len(m3):,} clients | achat répété : {m3['is_repeat_customer'].mean()*100:.2f} % | sans avis : {m3['review_missing'].mean()*100:.2f} %")

## 3. Balayage de *k* pour chaque variante
Pour chaque variante : variables plafonnées aux quantiles `CLIP_Q`, K-Means pour *k* de 2 à 8, silhouette, Davies-Bouldin, stabilité (ARI sur sous-échantillons de 80 %). Règle de choix : parmi *k* de 3 à 6 de stabilité ≥ 0,70, la **silhouette maximale** (sinon le plus stable).

In [ ]:
def prepare(features):
    X = m3[[c + "_rs" for c in features]].copy()
    X = X.clip(lower=X.quantile(CLIP_Q), upper=X.quantile(1 - CLIP_Q), axis=1)
    return X.values

def scan(Xv):
    sil_idx = rng.choice(len(Xv), size=min(SIL_SAMPLE, len(Xv)), replace=False)
    rows = []
    for k in K_RANGE:
        lab = KMeans(n_clusters=k, n_init=N_INIT, random_state=SEED).fit_predict(Xv)
        aris = []
        for b in range(N_BOOT):
            sub = rng.choice(len(Xv), size=int(0.8 * len(Xv)), replace=False)
            aris.append(adjusted_rand_score(lab, KMeans(n_clusters=k, n_init=3, random_state=SEED + b + 1).fit(Xv[sub]).predict(Xv)))
        rows.append({"k": k, "silhouette": silhouette_score(Xv[sil_idx], lab[sil_idx]), "davies_bouldin": davies_bouldin_score(Xv, lab),
                     "stabilité_ARI": float(np.mean(aris)), "taille_min_%": np.bincount(lab).min() / len(lab) * 100})
    return pd.DataFrame(rows).set_index("k")

def choose_k(sel):
    cand = sel.loc[(sel.index >= 3) & (sel.index <= 6)]
    stable = cand[cand["stabilité_ARI"] >= 0.70]
    return int((stable if len(stable) else cand)["silhouette" if len(stable) else "stabilité_ARI"].idxmax())

scans, chosen_k, X_var = {}, {}, {}
for v, spec in VARIANTS.items():
    X_var[v] = prepare(spec["features"])
    scans[v] = scan(X_var[v])
    chosen_k[v] = K_OVERRIDE.get(v, choose_k(scans[v]))
    print(f"\n{v} · {spec['nom']} : k retenu = {chosen_k[v]}")
    display(scans[v])

## 4. Que capture chaque variante ?
Pour chaque variante, au *k* retenu, la part de variance (**η²**, de 0 à 1) de quatre dimensions expliquée par l'appartenance au segment : récence, valeur, satisfaction, livraison. Un η² proche de 0 signifie que les segments ne distinguent pas cette dimension.

In [ ]:
DIMS = {"récence": "recency_days", "valeur": "log_monetary", "satisfaction": "avg_review_score", "livraison": "avg_delivery_days"}
def eta2(values, labels):
    tot = ((values - values.mean()) ** 2).sum()
    if tot == 0: return 0.0
    between = sum(((values[labels == c].mean() - values.mean()) ** 2) * (labels == c).sum() for c in np.unique(labels))
    return float(between / tot)

labels_var, rows = {}, []
for v, spec in VARIANTS.items():
    km = KMeans(n_clusters=chosen_k[v], n_init=N_INIT, random_state=SEED).fit(X_var[v])
    labels_var[v] = km.labels_
    sel = scans[v].loc[chosen_k[v]]
    row = {"variante": v, "description": spec["nom"], "k": chosen_k[v], "silhouette": sel["silhouette"], "davies_bouldin": sel["davies_bouldin"],
           "stabilité_ARI": sel["stabilité_ARI"], "plus petit segment (%)": sel["taille_min_%"]}
    for d, col in DIMS.items():
        row[f"η² {d}"] = eta2(m3[col].values, km.labels_)
    rows.append(row)
comp = pd.DataFrame(rows).set_index("variante")
display(comp)

eta_cols = [c for c in comp.columns if c.startswith("η²")]
comp[eta_cols].plot.bar(figsize=(10, 4), title="Part de la variance expliquée par les segments (η²)")
plt.ylabel("η²"); plt.xticks(rotation=0); plt.show()

## 5. Segmentation d'expérience retenue et dimension de valeur

In [ ]:
K = chosen_k[FINAL_VARIANT]
raw = labels_var[FINAL_VARIANT]
order = pd.Series(raw).value_counts().index.tolist()
m3["cluster"] = pd.Series(raw).map({old: new for new, old in enumerate(order)}).values
print(f"Segmentation retenue : {FINAL_VARIANT} ({VARIANTS[FINAL_VARIANT]['nom']}), k = {K}")
print(m3["cluster"].value_counts().sort_index().to_string())

# Dimensions de valeur et de récence : terciles du montant, médiane de la récence
m3["value_tier"] = pd.qcut(m3["monetary_total"].rank(method="first"), 3, labels=["valeur faible", "valeur moyenne", "valeur élevée"])
m3["recency_tier"] = np.where(m3["recency_days"] <= m3["recency_days"].median(), "récent", "ancien")
q = m3["monetary_total"].quantile([1 / 3, 2 / 3])
print(f"Seuils de valeur : faible <= {q.iloc[0]:.0f} BRL < moyenne <= {q.iloc[1]:.0f} BRL < élevée")

## 6. Noms et actions par segment d'expérience
Noms proposés à partir des écarts à la moyenne globale (satisfaction, retard), à relire : `LABEL_OVERRIDE` permet de les corriger.

In [ ]:
LABEL_OVERRIDE = {}      # par exemple {0: "Clients satisfaits"}
cols_z = ["recency_days", "log_monetary", "avg_review_score", "avg_delivery_days", "late_share"]
zf = (m3.groupby("cluster")[cols_z].mean() - m3[cols_z].mean()) / m3[cols_z].std()

def describe_cluster(c):
    r = zf.loc[c]
    unhappy, happy = r["avg_review_score"] < -0.5, r["avg_review_score"] > 0.3
    delivery = (r["late_share"] > 0.5) or (r["avg_delivery_days"] > 0.5)
    if unhappy and delivery: label = "Insatisfaits par la livraison"
    elif unhappy: label = "Insatisfaits (livraison normale)"
    elif happy: label = "Satisfaits"
    else: label = "Expérience moyenne"
    if delivery and not unhappy: label += ", livraison longue"
    return {"label": LABEL_OVERRIDE.get(c, label), "unhappy": bool(unhappy), "happy": bool(happy), "delivery": bool(delivery)}

info = {c: describe_cluster(c) for c in sorted(m3["cluster"].unique())}
m3["segment_experience"] = m3["cluster"].map({c: i["label"] for c, i in info.items()})

def action(c, tier):
    i = info[c]; hi, mid = tier == "valeur élevée", tier == "valeur moyenne"
    if i["unhappy"] and i["delivery"]:
        return "Geste commercial prioritaire et appel de suivi" if hi else "Message d'excuse et bon d'achat"
    if i["unhappy"]:
        return "Contact personnalisé et enquête qualité (produit, vendeur)" if hi else "Enquête de satisfaction"
    if i["happy"]:
        return "Programme de fidélité et offre de réachat premium" if hi else "Offre de réachat" if mid else "Communication standard"
    return "Suivi de la satisfaction ; communication proactive sur les délais" if i["delivery"] else "Communication standard"

m3["action"] = [action(c, t) for c, t in zip(m3["cluster"], m3["value_tier"])]
profile = m3.groupby("cluster").agg(clients=("customer_unique_id", "count"), note_moyenne=("avg_review_score", "mean"),
                                    duree_livraison_j=("avg_delivery_days", "mean"), part_en_retard=("late_share", "mean"),
                                    montant_median=("monetary_total", "median"), recence_mediane_j=("recency_days", "median"))
profile.insert(0, "segment", [info[c]["label"] for c in profile.index])
profile["part_clients_%"] = profile["clients"] / len(m3) * 100
display(profile)

## 7. Grille d'actions « expérience × valeur »
Nombre de clients, note moyenne et action recommandée pour chaque case. C'est la réponse au besoin de **ciblage par valeur** que la segmentation seule ne couvrait pas.

In [ ]:
grid_n = pd.crosstab(m3["segment_experience"], m3["value_tier"]).reindex(columns=["valeur faible", "valeur moyenne", "valeur élevée"])
grid_pct = (grid_n / len(m3) * 100).round(2)
grid_note = m3.pivot_table(index="segment_experience", columns="value_tier", values="avg_review_score", aggfunc="mean").reindex(columns=grid_n.columns)
print("Nombre de clients :"); display(grid_n)
print("Part des clients (%) :"); display(grid_pct)
print("Note moyenne :"); display(grid_note)

action_grid = (m3.groupby(["segment_experience", "value_tier"]).agg(clients=("customer_unique_id", "count"), action=("action", "first"))
                 .reset_index())
pd.set_option("display.max_colwidth", 90)
display(action_grid)

fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
sns.heatmap(grid_n, annot=True, fmt=",.0f", cmap="Blues", ax=ax[0]); ax[0].set_title("Clients par segment d'expérience et niveau de valeur")
sns.heatmap(grid_note, annot=True, fmt=".2f", cmap="RdYlGn", vmin=1, vmax=5, ax=ax[1]); ax[1].set_title("Note moyenne par case")
plt.tight_layout(); plt.show()

priority = m3[(m3["value_tier"] == "valeur élevée") & (m3["avg_review_score"] <= 2)]
late_priority = priority[priority["late_share"] > 0]
print(f"Clients de valeur élevée notés 2 ou moins : {len(priority):,} ({len(priority)/len(m3)*100:.2f} %), dont {len(late_priority):,} touchés par un retard")

## 8. Visualisation et profils régionaux

In [ ]:
Xv = X_var[FINAL_VARIANT]
pca = PCA(n_components=2, random_state=SEED).fit(Xv)
idx = rng.choice(len(Xv), size=min(8000, len(Xv)), replace=False); Z = pca.transform(Xv[idx])
fig, ax = plt.subplots(1, 2, figsize=(16, 5.5))
sc = ax[0].scatter(Z[:, 0], Z[:, 1], c=m3["cluster"].values[idx], cmap="tab10", s=6, alpha=0.6)
ax[0].legend(*sc.legend_elements(), title="cluster"); ax[0].set_title(f"Projection PCA ({pca.explained_variance_ratio_.sum()*100:.0f} % de variance)")
reg = pd.crosstab(m3["region"], m3["segment_experience"], normalize="index") * 100
reg.plot.bar(stacked=True, ax=ax[1], rot=20); ax[1].set_title("Répartition des segments par région (%)"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()
display(reg.round(1))

## 9. Fichiers de déploiement

In [ ]:
cols = ["customer_unique_id", "cluster", "segment_experience", "value_tier", "recency_tier", "action", "customer_state", "region",
        "n_orders", "monetary_total", "avg_order_value", "avg_review_score", "avg_delivery_days", "late_share", "recency_days"]
m3[cols].to_csv(os.path.join(OUT_DIR, "customer_segments_final.csv"), index=False)
profile.to_csv(os.path.join(OUT_DIR, "segment_profiles_final.csv"))
action_grid.to_csv(os.path.join(OUT_DIR, "segment_action_grid.csv"), index=False)
grid_n.to_csv(os.path.join(OUT_DIR, "segment_grid_counts.csv")); grid_note.to_csv(os.path.join(OUT_DIR, "segment_grid_notes.csv"))
comp.to_csv(os.path.join(OUT_DIR, "segmentation_variants_comparison.csv"))
json.dump({"variant": FINAL_VARIANT, "k": int(K), "features": VARIANTS[FINAL_VARIANT]["features"],
           "value_tier_thresholds_brl": [float(q.iloc[0]), float(q.iloc[1])], "recency_median_days": float(m3["recency_days"].median()),
           "high_value_unhappy_clients": int(len(priority)), "sizes": {int(k): int(v) for k, v in m3["cluster"].value_counts().sort_index().items()}},
          open(os.path.join(OUT_DIR, "segmentation_manifest.json"), "w"), ensure_ascii=False, indent=2)
print("Fichiers de déploiement :", sorted(os.listdir(OUT_DIR)))

## Synthèse à rédiger
- **Ce que capture chaque variante** (cellule 4) : η² de la récence, de la valeur, de la satisfaction et de la livraison.
- **Choix** : la segmentation d'expérience reste la plus stable et la plus interprétable ; la valeur est ajoutée par terciles (cellule 5).
- **Grille d'actions** « expérience × valeur » : cases prioritaires et effectifs (cellule 7).
- **Limite** : 97 % de clients à achat unique, donc une segmentation d'expérience et non de fidélité.